## 1. RNN（Recurrent Neural Network）

`RNN`（再帰型ニューラルネットワーク）は、**順序を持つデータを学習する深層学習モデル**である。

時系列データやテキストなどの<strong>順序データ（Sequential Data）</strong>から特徴を学習し、  
**分類・回帰の両方**に利用できる。

- **特徴**：前の時点の情報を、次の時点の学習に利用する。
- **主な用途**：株価予測、時系列分析、自然言語処理など
- **基本構造**：現在の入力と前の時点の隠れ状態を使って、新しい隠れ状態を計算する。
- **一般的なニューラルネットワークとの違い**：RNNは前の時点の情報を次の時点へ伝える。

<br>

#### ※ RNNにおける回帰・分類

<table style="
width:950px;
max-width:100%;
border-collapse:collapse;
color:#000;
text-align:left;
margin-left:0 !important;
margin-right:auto !important;
">
<tbody>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:10px 18px; width:180px; text-align:center; color:#000;">
<span style="color:#ff4d5a;"><strong>回帰</strong></span><br>
<strong>（Regression）</strong>
</td>

<td style="padding:10px 18px; color:#000; line-height:1.8;">
<strong>目的：</strong>連続した数値を予測する。<br>
<span style="color:#ff4d5a;">例：</span>翌日の株価、次週の気温、将来の電力使用量<br>
<strong>出力層：</strong>活性化関数なし（または Linear）のニューロンを使い、連続値を出力する。<br>
<strong>損失関数：</strong><span style="color:#ff8c00;"><strong>MSE・MAE</strong></span> などを使用する。
</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:10px 18px; width:180px; text-align:center; color:#000;">
<span style="color:#ff4d5a;"><strong>分類</strong></span><br>
<strong>（Classification）</strong>
</td>

<td style="padding:10px 18px; color:#000; line-height:1.8;">
<strong>目的：</strong>カテゴリやラベルを予測する。<br>
<span style="color:#ff4d5a;">例：</span>感情分析、スパムメール判定、音声コマンド認識<br>
<strong>出力層：</strong><span style="color:#ff8c00;"><strong>Softmax・Sigmoid</strong></span> を使ってクラスの確率を出力する。<br>
<strong>損失関数：</strong><span style="color:#ff8c00;"><strong>Cross-Entropy</strong></span> を使用する。
</td>
</tr>

</tbody>
</table>

<br>

#### ■ RNN予測の考え方

過去の一定期間のデータを使って、**次の時点の値を予測する**。

<br>

例)

**1～3日目の株価 → RNNで学習 → 4日目の株価を予測**

- 入力区間を1日ずつ移動させる**スライディングウィンドウ**を使って、複数の学習サンプルを作成する。

<br>

In [ ]:
import numpy as np

X = np.array([1, 2, 3, 4, 5, 6])

display(X.shape)             # (6,)

display(X.reshape(2, 3))     # 2行 × 3列

display(X.reshape(3, 2))     # 3行 × 2列

display(X.reshape(3, 2, 1))  # 3 × 2 × 1 の3次元配列

(6,)

array([[1, 2, 3],
       [4, 5, 6]])

array([[1, 2],
       [3, 4],
       [5, 6]])

array([[[1],
        [2]],

       [[3],
        [4]],

       [[5],
        [6]]])

<Br>

<br>

#### ■ スライディングウィンドウ

過去3日分を**入力データ**、その次の日を**正解データ**として学習サンプルを作る。

- `price[i:i+3]`：過去3日分の入力データ
- `price[i+3]`：次の日の正解データ
- `price[-3:]`：全データの最後の3日分

<span style="color:red;">
入力：過去3日 → 正解：次の1日
</span>

<br>
<br>

## 【RNNによる株価予測】

仮想の株価データを使って、**過去3日分の株価から次の日の株価を予測するRNN回帰モデル**を構築・学習する。

<span style="color:red;">
※ 株価の金額単位はすべて韓国ウォン（KRW）で表記する。
</span>

<br>

#### ① 株価データ準備

RNNの回帰を練習するために、仮想の株価データを用意する。

In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense

price = np.array([
    100, 102, 101, 105, 107, 106, 110, 112, 115, 113,
    118, 120, 119, 123, 125, 128, 126, 130, 133, 135
], dtype=float)

<Br>

#### ② 正規化

入力値を小さな範囲に変換し、ニューラルネットワークの学習を安定させる。

<br>

##### ※ 主な正規化方法

- **一定の値で割る**：今回のように `150` などで割って、値の大きさを小さくする。
- **MaxAbsScaler**：各値を**最大絶対値**で割り、通常 `-1～1` の範囲に変換する。
- **MinMaxScaler**：最小値を `0`、最大値を `1` として、**0～1の範囲**に変換する。

<span style="color:red;">
今回の <strong>/ 150</strong> は仕組みを理解するための簡単な方法であり、実際の前処理では <strong>Min-Max</strong> などのスケーリング方法も使用される。
</span>

In [3]:
price = price / 150

<br>

#### ③ 学習データ作成

**スライディングウィンドウ**を使って、過去3日分の株価から次の日の株価を予測する学習データを作る。

<span style="color:red;">
1～3日目 → 4日目を予測<br>
2～4日目 → 5日目を予測<br>
3～5日目 → 6日目を予測
</span>

→ 20日分のデータから、**17個の学習サンプル**を作成する。

In [ ]:
X, y = [], []

for i in range(len(price) - 3):
    X.append(price[i:i+3])  # 過去3日分
    y.append(price[i+3])    # 次の日の正解値

X = np.array(X)
y = np.array(y)

print('X.shape:', X.shape) 
print('y:', y)

X.shape: (17, 3)
y: [0.7        0.71333333 0.70666667 0.73333333 0.74666667 0.76666667
 0.75333333 0.78666667 0.8        0.79333333 0.82       0.83333333
 0.85333333 0.84       0.86666667 0.88666667 0.9       ]


<br>

#### ④ RNN入力形式に変換

RNNでは、入力データを基本的に**3次元形式**で扱う。

<span style="color:red;">
学習サンプル数 × 時系列の長さ × 特徴量数
</span>

<br> `reshape(-1, 3, 1)`

- `-1`：学習サンプル数を自動計算  
  `1`の場合：サンプル数を1個に固定
- `3`：過去3日分
- `1`：各時点の特徴量（株価）1個

→ `(17, 3)` を **`(17, 3, 1)`** に変換する。

In [5]:
X = X.reshape(-1, 3, 1)

<br>

#### ⑤ RNNモデル構築・学習設定

過去3日分の株価を入力し、**次の日の株価1個を予測する回帰モデル**を構築する。

In [6]:
model = Sequential([
    Input(shape=(3, 1)),  # 3時点 × 特徴量1個
    SimpleRNN(10),        # RNN層：10ニューロン
    Dense(1)              # 次の日の株価を1つ予測
])

model.compile(
    optimizer='adam',
    loss='mse'
)

<br>

#### ⑥ モデル学習

作成した17個の学習サンプルを使って、RNNモデルを学習する。

In [7]:
model.fit(
    X,
    y,
    epochs=100,  # 全学習データを100回繰り返す
    verbose=0
)

<br>

#### ⑦ 次の日の株価を予測

全データの**最後の3日分**を入力し、その次の日の株価を予測する。

<span style="color:red;">
この例には実際の「次の日」の株価がないため、予測値の正確さを評価することはできない。
</span>

In [8]:
last_days = price[-3:].reshape(1, 3, 1)

pred = model.predict(last_days, verbose=0)

# 正規化した値を元の株価単位に戻す
print(f"予測株価：{pred[0][0] * 150:.2f}")

予測株価：134.31


<Br>

#### ※ RNN株価予測の実習フロー

<table align="left" style="
width:950px;
max-width:100%;
border-collapse:collapse;
color:#000;
text-align:left;
margin-left:0 !important;
margin-right:auto !important;
">

<thead>
<tr style="background-color:#dfe3e8; color:#000;">
<th style="padding:8px 16px; width:70px; text-align:center;"><strong>段階</strong></th>
<th style="padding:8px 16px; width:280px; text-align:center;"><strong>作業</strong></th>
<th style="padding:8px 16px; text-align:center;"><strong>内容</strong></th>
</tr>
</thead>

<tbody>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>①</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>株価データ準備</strong></td>
<td style="padding:8px 16px; text-align:left;">RNN回帰の学習に使用する仮想の株価データを用意する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>②</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>正規化</strong></td>
<td style="padding:8px 16px; text-align:left;">株価の値を小さな範囲に変換し、学習を安定させる。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>③</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>学習データ作成</strong></td>
<td style="padding:8px 16px; text-align:left;">スライディングウィンドウで、過去3日 → 次の日の学習サンプルを作成する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>④</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>RNN入力形式に変換</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>reshape(-1, 3, 1)</code>で3次元形式に変換する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑤</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>RNNモデル構築・学習設定</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>SimpleRNN</code>と<code>Dense</code>で回帰モデルを構築し、学習方法を設定する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑥</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>モデル学習</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>fit()</code>で作成した学習データを使ってRNNを学習する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑦</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>次の日の株価予測</strong></td>
<td style="padding:8px 16px; text-align:left;">最後の3日分を入力し、<code>predict()</code>で次の日の株価を予測する。</td>
</tr>

</tbody>
</table>

<br clear="all">

<Br>

## 【実際の株価データを使ったRNN実習】

`yfinance`で実際の株価データを取得し、RNNで学習できる形に前処理して株価予測を行う。

<br>

#### ① ライブラリ読み込み

株価データの取得・前処理・RNNモデル構築・評価に必要なライブラリを読み込む。

In [30]:
import yfinance as yf
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense
from sklearn.metrics import r2_score

<br>

#### ② 株価データ取得

`yfinance`を使って、Samsung Electronics（`005930.KS`）の直近3年間の日次株価データを取得する。

In [20]:
df = yf.download(
    "005930.KS",
    period="3y",       # 直近3年間
    interval="1d",     # 日次データ
    auto_adjust=True,
    progress=False
)

df.head()

Price,Close,High,Low,Open,Volume
Ticker,005930.KS,005930.KS,005930.KS,005930.KS,005930.KS
Date,,,,,
2023-10-23,65024.511719,65689.967248,64834.381568,65309.706946,11625959
2023-10-24,65119.574219,65404.769434,64359.053644,65309.704362,12791710
2023-10-25,64644.242188,65404.762684,64549.177125,65404.762684,10610703
2023-10-26,63408.398438,64549.179219,63408.398438,63693.593633,15517624
2023-10-27,63978.792969,63978.792969,63408.402541,63788.662826,11334726


<br>

#### ③ 必要なデータを選択

取得した株価データから、<strong>日付と終値</strong>だけを使用する。

In [ ]:
# 日付と終値だけを選択
df = df.reset_index()

df = df[["Date", "Close"]]
df.columns = ["日付", "終値"]
df = df.reset_index(drop=True)

display(df.head())

,日付,終値
0,2023-10-23,65024.511719
1,2023-10-24,65119.574219
2,2023-10-25,64644.242188
3,2023-10-26,63408.398438
4,2023-10-27,63978.792969


<Br>

#### ④ 欠損値確認

学習前に、株価データに欠損値が含まれていないか確認する。

In [22]:
display(df.isna().sum())

日付    0
終値    0
dtype: int64

<br>

#### ⑤ NumPy配列に変換

終値データを取り出し、`MinMaxScaler`で処理できる**2次元配列**に変換する。

In [23]:
data = df["終値"].to_numpy().reshape(-1, 1)

print("data.shape:", data.shape)

data.shape: (730, 1)


<br>

#### ⑥ 株価データ正規化

`MinMaxScaler`を使って、終値を **0～1の範囲**に正規化する。

<span style="color:red;">
最小値 → 0、最大値 → 1
</span>

In [26]:
print("正規化前")
print(data[:5])

scaler = MinMaxScaler()
data = scaler.fit_transform(data)

print("正規化後")
print(data[:5])

正規化前
[[65024.51171875]
 [65119.57421875]
 [64644.2421875 ]
 [63408.3984375 ]
 [63978.79296875]]
正規化後
[[0.0531093 ]
 [0.0534123 ]
 [0.05189724]
 [0.04795816]
 [0.04977621]]


<br>

#### ⑦ RNN学習データ作成

**スライディングウィンドウ**を使って、過去20日分の終値から次の日の終値を予測する学習データを作成する。

<span style="color:red;">
入力：過去20日分 → 正解：次の1日
</span>

- `X`：過去20日分の終値
- `y`：その次の日の終値
- RNN入力形式：**サンプル数 × 20日 × 特徴量1個**

In [27]:
X = []
y = []

for i in range(len(data) - 20):
    X.append(data[i:i+20])   # 過去20日分の終値
    y.append(data[i+20, 0])  # 次の日の終値

X = np.array(X)
y = np.array(y)

display(X.shape)  # (サンプル数, 20, 1)
display(y.shape)  # (サンプル数,)

print(X[0])
print(y[0])

(710, 20, 1)

(710,)

[[0.0531093 ]
 [0.0534123 ]
 [0.05189724]
 [0.04795816]
 [0.04977621]
 [0.04977621]
 [0.04856414]
 [0.05371534]
 [0.0570484 ]
 [0.0567454 ]
 [0.06068444]
 [0.06068444]
 [0.05765438]
 [0.0588664 ]
 [0.05947242]
 [0.05916942]
 [0.06038147]
 [0.06462358]
 [0.06644162]
 [0.06553258]]
0.0661386245418269


<br>

#### ⑧ 学習データ・テストデータ分割

時系列の順番を維持したまま、データを以下のように分割する。

- **前80%**：学習データ
- **後20%**：テストデータ

<span style="color:red;">
時系列データなので `shuffle=False` とし、データの順番を崩さない。
</span>

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

print(X_train.shape)
print(X_test.shape)

(568, 20, 1)
(142, 20, 1)


<br>

#### ⑨ RNNモデル構築・学習設定

過去20日分の終値を入力し、**次の日の終値1個を予測する回帰モデル**を構築する。

In [ ]:
model = Sequential([
    Input(shape=(20, 1)),  # 20日 × 特徴量1個
    SimpleRNN(50),         # RNN層：50ニューロン
    Dense(1)               # 次の日の終値を1つ予測
])

model.compile(
    optimizer="adam",  # 重みを更新するアルゴリズム
    loss="mse"         # 回帰の損失関数
)

<Br>

#### ⑩ モデル学習

学習データを32サンプルずつ処理し、全データを20回繰り返して学習する。

In [32]:
model.fit(
    X_train,
    y_train,
    epochs=20,       # 全学習データを20回繰り返す
    batch_size=32,   # 1回に32サンプルずつ学習
    shuffle=False    # 時系列の順番を維持
)

Epoch 1/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.0043 
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0066
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0018   
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 6.3056e-04
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 6.1771e-04
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 4.5351e-04
Epoch 7/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5.1559e-04
Epoch 8/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 7.3668e-04
Epoch 9/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0012    
Epoch 10/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0021    
Epoch 11/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0012    
Epoch 12/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 8.0138e-04
Epoch 13/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4.5115e-04
Epoch 14/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4.3903e-04 
Epoch 15/20
18/18 ━━━━━━━━━━━━━━━━━━

<Br>

#### ⑪ テストデータ予測・評価

テストデータから終値を予測し、正規化された値を元の株価単位に戻して実際の終値と比較する。

In [ ]:
pred = model.predict(X_test)

# 正規化された予測値を元の株価単位に戻す
pred = scaler.inverse_transform(pred)

# y_testを2次元に変換して元の株価単位に戻す
actual = scaler.inverse_transform(
    y_test.reshape(-1, 1)
)

result = pd.DataFrame({
    "実際の終値": actual.flatten(),
    "予測終値": pred.flatten()
})

display(result)

r2 = r2_score(actual, pred)
print(f"R² Score：{r2:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step


,実際の終値,予測終値
0,179530.015625,189269.203125
1,179530.015625,192079.984375
2,176105.781250,191399.484375
3,167015.812500,186574.234375
4,189441.078125,170818.343750
...,...,...
137,218000.000000,214835.593750
138,211500.000000,207522.375000
139,223500.000000,207332.375000
140,214500.000000,210836.937500


R² Score：0.6676


<br>

#### ⑫ 次の取引日の終値予測

最新の**20営業日分の終値**を入力し、その次の取引日の終値を予測する。

In [36]:
last_20_days = data[-20:].reshape(1, 20, 1)

next_price = model.predict(
    last_20_days,
    verbose=0
)

# 元の株価単位に戻す
next_price = scaler.inverse_transform(next_price)

display(pd.DataFrame({
    "予測対象": ["次の取引日"],
    "予測終値": next_price.flatten()
}))

,予測対象,予測終値
0,次の取引日,212242.34375


<br>

#### ※ 実際の株価データを使ったRNN実習フロー

<table align="left" style="
width:950px;
max-width:100%;
border-collapse:collapse;
color:#000;
text-align:left;
margin-left:0 !important;
margin-right:auto !important;
">

<thead>
<tr style="background-color:#dfe3e8; color:#000;">
<th style="padding:8px 16px; width:70px; text-align:center;"><strong>段階</strong></th>
<th style="padding:8px 16px; width:290px; text-align:center;"><strong>作業</strong></th>
<th style="padding:8px 16px; text-align:center;"><strong>内容</strong></th>
</tr>
</thead>

<tbody>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>①</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>株価データ準備</strong></td>
<td style="padding:8px 16px; text-align:left;">株価予測に必要なライブラリを読み込み、実習の準備をする。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>②</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>株価データ取得</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>yfinance</code>でSamsung Electronicsの直近3年間の日次株価を取得する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>③</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>必要なデータを選択</strong></td>
<td style="padding:8px 16px; text-align:left;">取得したデータから、日付と終値だけを使用する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>④</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>欠損値確認</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>isna().sum()</code>で欠損値の有無を確認する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑤</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>NumPy配列に変換</strong></td>
<td style="padding:8px 16px; text-align:left;">終値を取り出し、学習に使いやすい2次元のNumPy配列に変換する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑥</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>株価データ正規化</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>MinMaxScaler</code>で終値を0～1の範囲に正規化する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑦</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>RNN学習データ作成</strong></td>
<td style="padding:8px 16px; text-align:left;">過去20営業日分の終値から、次の日の終値を予測する学習サンプルを作成する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑧</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>学習・テストデータ分割</strong></td>
<td style="padding:8px 16px; text-align:left;">時系列順を維持したまま、前80%を学習用、後20%をテスト用に分割する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑨</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>RNNモデル構築・学習設定</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>SimpleRNN</code>と<code>Dense</code>で回帰モデルを構築し、Adam・MSEを設定する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑩</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>モデル学習</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>fit()</code>で学習データを使ってRNNモデルを学習する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑪</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>テストデータ予測・評価</strong></td>
<td style="padding:8px 16px; text-align:left;">予測値を元の株価単位に戻し、実際の終値と比較してR²で性能を確認する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑫</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>次の取引日の終値予測</strong></td>
<td style="padding:8px 16px; text-align:left;">最新20営業日分の終値を入力し、<code>predict()</code>で次の取引日の終値を予測する。</td>
</tr>

</tbody>
</table>

<br clear="all">